# ДЗ-1: распознавание английской речи (CTC)
Включите GPU: **Среда выполнения → Сменить среду выполнения → T4 GPU**.
Вставьте ссылку на ZIP/TAR-архив с папкой `audio/` и, при наличии, `transcriptions/`.
Затем выполните **Среда выполнения → Выполнить все**. Остальные ячейки менять не нужно.
Код и веса загружаются автоматически. Если эталонов нет, сохраняются только предсказания.


In [ ]:
DATASET_URL = "" #@param {type:"string"}
REPOSITORY_URL = "https://github.com/NS1973/DLA-ASR-2026.git"


## 1. Установка проекта
Клонируем сдаваемую ветку и устанавливаем зафиксированные зависимости.

In [ ]:
import os, subprocess, sys
from pathlib import Path
PROJECT = Path('/content/DLA-ASR-2026')
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch','codex/hw1-asr',REPOSITORY_URL,str(PROJECT)],check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import torch
print('PyTorch:',torch.__version__,'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Загрузка модели
Скрипт проверяет SHA-256 весов по манифесту сдачи.

In [ ]:
import json
subprocess.run([sys.executable,'scripts/download_resources.py'],check=True)
assert Path('artifacts/model.pt').is_file()
manifest = json.loads(Path('resources.json').read_text())
INFERENCE_OPTIONS = manifest.get('inference_overrides', ['decoder=greedy'])
print('Настройки сдаваемой модели:', INFERENCE_OPTIONS)


## 3. Пример использования
Небольшой настоящий пример показывает инференс и отдельный расчёт метрик.

In [ ]:
subprocess.run([sys.executable,'inference.py','checkpoint=artifacts/model.pt','custom_dir=artifacts/example','output_dir=predictions/example','workers=0',*INFERENCE_OPTIONS],check=True)
subprocess.run([sys.executable,'calc_metrics.py','--reference-dir','artifacts/example/transcriptions','--prediction-dir','predictions/example'],check=True)
from IPython.display import Audio, display
audio = next(Path('artifacts/example/audio').iterdir())
display(Audio(filename=str(audio)))
uid = audio.stem
print('Эталон:', Path(f'artifacts/example/transcriptions/{uid}.txt').read_text().strip())
print('Распознано:', Path(f'predictions/example/{uid}.txt').read_text().strip())


## 4. Ваш набор данных
Поддерживаются прямая ссылка на архив и публичная ссылка Google Drive. Архив может содержать одну внешнюю папку.

In [ ]:
if not DATASET_URL.strip():
    raise ValueError('Вставьте ссылку на ваш набор в поле DATASET_URL первой ячейки.')
from tempfile import mkdtemp
Path('data').mkdir(exist_ok=True)
USER_WORKSPACE = Path(mkdtemp(prefix='user-',dir='data')).resolve()
USER_PREDICTIONS = USER_WORKSPACE / 'predictions'
subprocess.run([sys.executable,'scripts/prepare_custom_dataset.py','--url',DATASET_URL,'--output-dir',str(USER_WORKSPACE/'dataset')],check=True)
dataset = (USER_WORKSPACE/'dataset/dataset_path.txt').read_text(encoding='utf-8').strip()
subprocess.run([sys.executable,'inference.py','checkpoint=artifacts/model.pt',f'custom_dir={dataset}',f'output_dir={USER_PREDICTIONS}','workers=0',*INFERENCE_OPTIONS],check=True)
if (Path(dataset)/'transcriptions').is_dir():
    subprocess.run([sys.executable,'calc_metrics.py','--reference-dir',str(Path(dataset)/'transcriptions'),'--prediction-dir',str(USER_PREDICTIONS),'--output',str(USER_PREDICTIONS/'metrics.json')],check=True)
else:
    print('Эталонов нет: метрики не рассчитываются. Предсказания:', USER_PREDICTIONS)


## 5. Скачать результат

In [ ]:
import shutil
shutil.make_archive('/content/asr_predictions','zip',str(USER_PREDICTIONS))
from google.colab import files
files.download('/content/asr_predictions.zip')
